# Prepare the Kaggle dataset

This notebook downloads the Indonesian License Plate Dataset, isolates its detection annotations, converts the source labels to detector-compatible YOLO rows, creates a reproducible train/validation/test split, and validates the result for the training notebook.

The archive also contains a recognition subset. It is intentionally excluded here; we will use that later for an OCR model. The detection labels in the archive include an optional sixth plate-text field, which is removed from the detector copy.

In [ ]:
from pathlib import Path
import hashlib
import os
import random
import shutil
import zipfile

import requests
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Google Drive must already be mounted if you use the default path.")

DRIVE_ROOT = Path("/content/drive/MyDrive/indonesia-license-plate-model")
DATASET_DIR = DRIVE_ROOT / "dataset_yolo"
DOWNLOAD_DIR = Path("/content/indonesia-license-plate-kaggle")
ARCHIVE_PATH = DOWNLOAD_DIR / "indonesian-license-plate-dataset-v1.zip"
EXTRACT_DIR = DOWNLOAD_DIR / "extracted"

KAGGLE_DOWNLOAD_URL = "https://www.kaggle.com/api/v1/datasets/download/juanthomaswijaya/indonesian-license-plate-dataset?dataset_version_number=1"
SEED = 42
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}
SPLITS = ("train", "val", "test")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def dataset_is_complete(dataset_dir):
    """Return True only when every split has paired image and label files."""
    if not dataset_dir.exists():
        return False
    for split in SPLITS:
        image_dir = dataset_dir / "images" / split
        label_dir = dataset_dir / "labels" / split
        if not image_dir.is_dir() or not label_dir.is_dir():
            return False
        image_stems = {path.stem for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS}
        label_stems = {path.stem for path in label_dir.glob("*.txt")}
        if not image_stems or image_stems != label_stems:
            return False
    return True

dataset_ready = dataset_is_complete(DATASET_DIR)
if dataset_ready:
    print(f"Prepared dataset already exists at {DATASET_DIR}; it will not be overwritten.")
elif DATASET_DIR.exists():
    print(f"Prepared dataset at {DATASET_DIR} is incomplete; it will be rebuilt.")
    print("The incomplete folder will be backed up instead of deleted.")
else:
    print(f"Prepared dataset will be created at {DATASET_DIR}")

# Optional authentication. In Colab, add KAGGLE_API_TOKEN to Colab Secrets, or set
# KAGGLE_USERNAME and KAGGLE_KEY in the runtime environment. Public downloads may work
# without credentials, depending on Kaggle's current access policy.
headers = {}
auth = None
try:
    from google.colab import userdata
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    kaggle_token = os.environ.get("KAGGLE_API_TOKEN")
if kaggle_token:
    headers["Authorization"] = f"Bearer {kaggle_token}"
elif os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
    auth = (os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"])

if not ARCHIVE_PATH.exists():
    print("Downloading Kaggle archive...")
    response = requests.get(KAGGLE_DOWNLOAD_URL, headers=headers, auth=auth, stream=True, timeout=120)
    response.raise_for_status()
    total_bytes = int(response.headers.get("content-length", 0))
    with ARCHIVE_PATH.open("wb") as output:
        progress = tqdm(total=total_bytes, unit="B", unit_scale=True, desc="dataset")
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)
                progress.update(len(chunk))
        progress.close()
else:
    print(f"Using existing archive: {ARCHIVE_PATH}")

if not zipfile.is_zipfile(ARCHIVE_PATH):
    raise ValueError(
        "The Kaggle response was not a ZIP archive. Check Kaggle access or add KAGGLE_API_TOKEN to Colab Secrets."
    )
if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.rglob("*")):
    print("Extracting archive into temporary Colab storage...")
    shutil.unpack_archive(str(ARCHIVE_PATH), str(EXTRACT_DIR))
print("Archive ready:", ARCHIVE_PATH)
print("Extracted to:", EXTRACT_DIR)

In [ ]:
all_images = [path for path in EXTRACT_DIR.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]
image_index = {}
for image_path in all_images:
    image_index.setdefault(image_path.stem, []).append(image_path)

label_paths = [
    path for path in EXTRACT_DIR.rglob("*.txt")
    if "recogn" not in str(path).lower() and "ocr" not in str(path).lower()
]


def find_image_for_label(label_path):
    direct_matches = [
        label_path.with_suffix(extension)
        for extension in IMAGE_EXTENSIONS
        if label_path.with_suffix(extension).exists()
    ]
    if direct_matches:
        return direct_matches[0]
    stem_matches = [
        image_path for image_path in image_index.get(label_path.stem, [])
        if "recogn" not in str(image_path).lower() and "ocr" not in str(image_path).lower()
    ]
    if not stem_matches:
        return None
    same_parent = [path for path in stem_matches if path.parent == label_path.parent]
    return sorted(same_parent or stem_matches, key=lambda path: str(path))[0]


def sha1(path):
    digest = hashlib.sha1()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_detection_label(label_path):
    """Keep class and normalized box fields; drop an optional plate-text field."""
    normalized_lines = []
    for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
        fields = line.split()
        if len(fields) not in (5, 6):
            raise ValueError(f"Unexpected label format in {label_path} line {line_number}: {line}")
        normalized_lines.append(" ".join(fields[:5]))
    return "\n".join(normalized_lines) + ("\n" if normalized_lines else "")

pairs = []
unmatched_labels = []
for label_path in label_paths:
    image_path = find_image_for_label(label_path)
    if image_path is None:
        unmatched_labels.append(label_path)
    else:
        pairs.append((image_path, label_path))

# Remove duplicate image files while keeping the first matching annotation file.
unique_pairs = {}
for image_path, label_path in pairs:
    unique_pairs.setdefault(sha1(image_path), (image_path, label_path))
pairs = list(unique_pairs.values())

print(f"Detection label files found: {len(label_paths)}")
print(f"Matched image/label pairs:   {len(pairs)}")
print(f"Unmatched labels:            {len(unmatched_labels)}")
if unmatched_labels:
    print("First unmatched labels:", unmatched_labels[:5])
if len(pairs) < 3:
    raise ValueError("Fewer than three detection image/label pairs were found; inspect the extracted archive layout.")

In [ ]:
random.Random(SEED).shuffle(pairs)

if not dataset_ready:
    split_count = len(pairs)
    train_end = max(1, int(split_count * SPLIT_RATIOS["train"]))
    val_end = min(split_count - 1, train_end + max(1, int(split_count * SPLIT_RATIOS["val"])))
    split_pairs = {
        "train": pairs[:train_end],
        "val": pairs[train_end:val_end],
        "test": pairs[val_end:],
    }

    staging_dir = Path("/content/indonesia-license-plate-prepared")
    if staging_dir.exists():
        shutil.rmtree(staging_dir)
    for split, split_items in split_pairs.items():
        image_dir = staging_dir / "images" / split
        label_dir = staging_dir / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        for index, (image_path, label_path) in enumerate(split_items):
            output_stem = f"{index:06d}_{image_path.stem}"
            shutil.copy2(image_path, image_dir / f"{output_stem}{image_path.suffix.lower()}")
            output_label = label_dir / f"{output_stem}.txt"
            output_label.write_text(normalize_detection_label(label_path))

    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DATASET_DIR.exists():
        backup_dir = DATASET_DIR.with_name(f"{DATASET_DIR.name}_incomplete_backup")
        backup_index = 2
        while backup_dir.exists():
            backup_dir = DATASET_DIR.with_name(f"{DATASET_DIR.name}_incomplete_backup_{backup_index}")
            backup_index += 1
        shutil.move(str(DATASET_DIR), str(backup_dir))
        print(f"Backed up incomplete dataset to {backup_dir}")
    shutil.copytree(staging_dir, DATASET_DIR)
    print(f"Prepared {len(pairs)} unique images with seed {SEED}.")
    for split, split_items in split_pairs.items():
        print(f"{split:5}: {len(split_items)} images")
else:
    print("Skipped preparation because DATASET_DIR is complete.")

In [ ]:
from collections import Counter

bad_rows = []
object_counts = Counter()
for split in ("train", "val", "test"):
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    image_paths = sorted(path for path in image_dir.glob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)
    label_paths = sorted(label_dir.glob("*.txt"))
    image_stems = {path.stem for path in image_paths}
    label_stems = {path.stem for path in label_paths}
    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)
    print(f"{split:5}: images={len(image_paths):4} labels={len(label_paths):4}")
    if missing_labels or missing_images:
        raise ValueError(f"{split} pairing error: missing labels={missing_labels[:3]}, missing images={missing_images[:3]}")

    for label_path in label_paths:
        for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
            fields = line.split()
            if len(fields) != 5:
                bad_rows.append((label_path, line_number, "expected 5 fields"))
                continue
            try:
                class_id, x_center, y_center, width, height = map(float, fields)
            except ValueError:
                bad_rows.append((label_path, line_number, "non-numeric value"))
                continue
            if class_id != 0 or any(value < 0 or value > 1 for value in (x_center, y_center, width, height)):
                bad_rows.append((label_path, line_number, "class must be 0 and coordinates must be normalized"))
            object_counts[int(class_id)] += 1

print("Objects by class:", dict(object_counts))
if bad_rows:
    print("Invalid label rows (first 10):")
    for row in bad_rows[:10]:
        print(row)
    raise ValueError(f"Found {len(bad_rows)} invalid label rows")
print("Dataset is ready for 02_train_detector.ipynb.")

### Result

The prepared dataset is stored at:

    MyDrive/indonesia-license-plate-model/dataset_yolo/
    ├── images/{train,val,test}/
    └── labels/{train,val,test}/

The Kaggle page lists the dataset license as Unknown. Confirm the uploader's usage terms before distributing a trained model or using it commercially.